# Natural Language Processing Lab
## Experiment 2: Text Cleaning Using Python and Regular Expressions
**Domain Application:** Terms & Conditions Summarizer (Legal Document Normalization & Cleaning)

### Step 1: Ingest Raw HTML Document
Load the scraped legal document containing HTML structure, metadata tags, hyperlinks, and contact info.

In [1]:
import re
from bs4 import BeautifulSoup
import pandas as pd

# Read the raw scraped HTML contract
with open("data/dirty_tc_sample.html", "r", encoding="utf-8") as file:
    raw_html_content = file.read()

print("--- Raw HTML Document Snippet (First 400 chars) ---")
print(raw_html_content[:400])

--- Raw HTML Document Snippet (First 400 chars) ---
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Raw Scraped Terms of Service</title>
</head>
<body>
    <div class="legal-container">
        <h1>Terms and Conditions &amp; Privacy Notice</h1>
        <p class="meta">Effective Date: <b>January 15, 2026</b> &bull; Version 4.2.1 &copy; All Rights Reserved.</p>
        
        <div class="clause-section" id="sec-1">
   


### Step 2: Modular Text Cleaning Functions using `re` and `BeautifulSoup`
Implement isolated operations: HTML tag stripping, URL removal, email removal, and whitespace normalization.

In [2]:
# 1. Strip HTML tags
def remove_html_tags(text):
    if not isinstance(text, str):
        return ""
    return BeautifulSoup(text, "html.parser").get_text(separator=" ")

# 2. Remove URLs
def remove_urls(text):
    return re.sub(r"https?://\S+|www\.\S+", " ", text)

# 3. Remove Email Addresses
def remove_emails(text):
    return re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", " ", text)

# 4. Remove Phone numbers
def remove_phone_numbers(text):
    return re.sub(r"\+?\d{1,3}[-\s]?\(?\d{3}\)?[-\s]?\d{3}[-\s]?\d{4}", " ", text)

# 5. Normalize whitespace
def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

# Test step-by-step pipeline
text_step1 = remove_html_tags(raw_html_content)
text_step2 = remove_urls(text_step1)
text_step3 = remove_emails(text_step2)
text_step4 = remove_phone_numbers(text_step3)
text_cleaned = normalize_whitespace(text_step4)

print("--- Extracted & Cleaned Text ---")
print(text_cleaned)

--- Extracted & Cleaned Text ---
Raw Scraped Terms of Service Terms and Conditions & Privacy Notice Effective Date: January 15, 2026 • Version 4.2.1 © All Rights Reserved. 1. User Account & Eligibility You must be at least 18 years of age to access our services. If you encounter issues, contact our support team at or visit our portal at . Call helpline: for immediate assistance. 2. Limitation of Liability & Disclaimers SERVICES ARE PROVIDED "AS IS" AND "AS AVAILABLE" WITH ALL FAULTS. UNDER NO CIRCUMSTANCE SHALL THE COMPANY BE LIABLE FOR ANY AMOUNT EXCEEDING $500.00 USD OR 100% OF TOTAL FEES PAID IN THE PAST 12 MONTHS. We do NOT warrant that the service will be uninterrupted, error-free, or free from malicious viruses. 3. Termination & Suspension Section 3.1(b): We reserve the right to immediately terminate or suspend your access without notice if we suspect fraudulent activity or breach of agreement. For dispute arbitration details, see .


### Step 3: Standard Aggressive Cleaning vs Legal-Aware Cleaning
Highlight the problem of **over-cleaning** in legal contracts (preserving negation words, monetary values, percentages, and section indices).

In [3]:
# Standard Aggressive Cleaner (Strips all numbers and non-alpha characters)
def standard_aggressive_clean(text):
    text = remove_html_tags(text).lower()
    text = remove_urls(text)
    text = remove_emails(text)
    text = re.sub(r"\d+", " ", text)               # Strips all numbers
    text = re.sub(r"[^a-z\s]", " ", text)          # Strips all punctuation
    return normalize_whitespace(text)

# Legal-Aware Cleaner (Preserves crucial numbers, percentages, currency, and negation)
def legal_aware_clean(text):
    text = remove_html_tags(text)
    text = remove_urls(text)
    text = remove_emails(text)
    text = remove_phone_numbers(text)
    
    # Remove boilerplate symbols (e.g. copyright, bullets) but retain currency ($/EUR/GBP), %, and clause numbering
    text = re.sub(r"[\u00a9\u00ae\u2122\u2022*&]", " ", text)
    
    # Remove decorative quotes and escape slashes
    text = re.sub(r"[\"'\\]", "", text)
    
    return normalize_whitespace(text)

sample_liability = 'UNDER NO CIRCUMSTANCE SHALL THE COMPANY BE LIABLE FOR ANY AMOUNT EXCEEDING $500.00 USD OR 100% OF FEES PAID. We do NOT warrant error-free operation.'

print("--- Original Liability Clause ---")
print(sample_liability)

print("\n--- 1. Standard Aggressive Cleaning Output (Loses $500, 100%, casing) ---")
print(standard_aggressive_clean(sample_liability))

print("\n--- 2. Legal-Aware Cleaning Output (Retains amounts, percentages, and critical negation) ---")
print(legal_aware_clean(sample_liability))

--- Original Liability Clause ---
UNDER NO CIRCUMSTANCE SHALL THE COMPANY BE LIABLE FOR ANY AMOUNT EXCEEDING $500.00 USD OR 100% OF FEES PAID. We do NOT warrant error-free operation.

--- 1. Standard Aggressive Cleaning Output (Loses $500, 100%, casing) ---
under no circumstance shall the company be liable for any amount exceeding usd or of fees paid we do not warrant error free operation

--- 2. Legal-Aware Cleaning Output (Retains amounts, percentages, and critical negation) ---
UNDER NO CIRCUMSTANCE SHALL THE COMPANY BE LIABLE FOR ANY AMOUNT EXCEEDING $500.00 USD OR 100% OF FEES PAID. We do NOT warrant error-free operation.


### Step 4: Batch Cleaning on T&C Dataset (`tc_clauses.csv`)
Apply cleaning across the entire dataset and analyze noise reduction and token compression.

In [4]:
# Load clauses dataset from Experiment 1
df_clauses = pd.read_csv("data/tc_clauses.csv")

# Apply both cleaning methods
df_clauses["aggressive_clean"] = df_clauses["clause_text"].apply(standard_aggressive_clean)
df_clauses["legal_clean"] = df_clauses["clause_text"].apply(legal_aware_clean)

# Calculate character and word reductions
df_clauses["raw_len"] = df_clauses["clause_text"].str.len()
df_clauses["cleaned_len"] = df_clauses["legal_clean"].str.len()
df_clauses["noise_reduction_pct"] = ((df_clauses["raw_len"] - df_clauses["cleaned_len"]) / df_clauses["raw_len"]) * 100

print("--- Sample Cleaned Dataset (First 5 Rows) ---")
display(df_clauses[["clause_id", "company", "category", "clause_text", "legal_clean", "noise_reduction_pct"]].head())

print(f"\nAverage Noise Reduction across dataset: {df_clauses['noise_reduction_pct'].mean():.2f}%")

--- Sample Cleaned Dataset (First 5 Rows) ---


,clause_id,company,category,clause_text,legal_clean,noise_reduction_pct
0,AMZ_001,Amazon,Privacy,"Please review our Privacy Notice, which also g...","Please review our Privacy Notice, which also g...",0.0
1,AMZ_002,Amazon,Electronic Communications,You consent to receive communications from us ...,You consent to receive communications from us ...,0.0
2,AMZ_003,Amazon,Intellectual Property,All content included in or made available thro...,All content included in or made available thro...,0.0
3,AMZ_004,Amazon,Account Security,You are responsible for maintaining the confid...,You are responsible for maintaining the confid...,0.0
4,AMZ_005,Amazon,Termination,"Amazon reserves the right to refuse service, t...","Amazon reserves the right to refuse service, t...",0.0



Average Noise Reduction across dataset: 0.00%


### Step 5: Extension Activity – Regex Pattern Matcher for High-Risk T&C Provisions
Build regex extractors to automatically detect critical legal risk patterns in contracts.

In [5]:
# Extension Activity: Regex Legal Pattern Extractor
def extract_legal_risk_triggers(text):
    results = {}
    
    # 1. Termination & Suspension triggers
    results["termination_clauses"] = re.findall(r"[^.]*?\b(?:terminate|suspend|ban|cancel accounts?)\b[^.]*\.", text, re.IGNORECASE)
    
    # 2. Disclaimer & \"AS IS\" limitations
    results["liability_disclaimers"] = re.findall(r"[^.]*?\b(?:AS IS|NO LIABILITY|DISCLAIMS? ALL WARRANTIES|NOT BE LIABLE)\b[^.]*\.", text, re.IGNORECASE)
    
    # 3. Monetary limits & liability caps
    results["monetary_caps"] = re.findall(r"\$\d+(?:,\d+)*(?:\.\d+)?|\b\d+%(?:\s+OF\s+[A-Z]+)?", text, re.IGNORECASE)
    
    # 4. Arbitration & dispute venue mentions
    results["arbitration_clauses"] = re.findall(r"[^.]*?\b(?:arbitration|jury trial|dispute resolution|jurisdiction)\b[^.]*\.", text, re.IGNORECASE)
    
    return results

# Test extractor on combined scraped terms and Amazon/Alibaba clauses
combined_text = text_cleaned + " " + " ".join(df_clauses["clause_text"].tolist())
extracted_risks = extract_legal_risk_triggers(combined_text)

print("=== Automated Legal Pattern Extraction ===\n")
for category, matches in extracted_risks.items():
    print(f"[{category.upper()}] (Found: {len(matches)})")
    for m in matches[:2]:  # Show first 2 examples
        print("  ->", m.strip())
    print()

=== Automated Legal Pattern Extraction ===

[TERMINATION_CLAUSES] (Found: 3)
  -> 1(b): We reserve the right to immediately terminate or suspend your access without notice if we suspect fraudulent activity or breach of agreement.
  -> Amazon reserves the right to refuse service, terminate accounts, terminate your rights to use Amazon Services, remove or edit content, or cancel orders in its sole discretion.

[LIABILITY_DISCLAIMERS] (Found: 6)
  -> Limitation of Liability & Disclaimers SERVICES ARE PROVIDED "AS IS" AND "AS AVAILABLE" WITH ALL FAULTS.
  -> Amazon takes no responsibility and assumes no liability for any content posted by you or any third party.

[MONETARY_CAPS] (Found: 2)
  -> $500.00
  -> 100% OF TOTAL

[ARBITRATION_CLAUSES] (Found: 3)
  -> For dispute arbitration details, see .
  -> Each party waives any right to a jury trial and agrees that any dispute resolution proceedings will be conducted only on an individual basis.

